## Cleaning steps

1. Remove duplicates
2. Remove all row nulls
3. Fill the nulls/''/' '  
4. Trim spaces
5. Standardize case
6. Cast data types
7. Filter invalid rows
7. Rename columns
9. Add metadata columns if applicable
10. Validate schema

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

## Load table


In [0]:
df = spark.table('bikes_catalog.bronze.bronze_prd_info')
df.show()

## Cleaning


In [0]:
df = df.drop('prd_id')
df.show()

In [0]:
df = df.dropDuplicates()
df = df.dropna(how='all')

In [0]:
for col in df.columns:
    print(f'Nulls in {col}: {df.filter(F.col(col).isNull()).count()}')

In [0]:
df = df.na.fill({'prd_line': 'Unknown'}) 

df = df.withColumn(  #as it's not a string
    'prd_cost',
    F.when(df.prd_cost.isNull(), F.lit(0))
    .otherwise(df.prd_cost)
)
df = df.withColumn(  
    'prd_end_dt',
    F.when(df.prd_end_dt.isNull(), F.lit('9999-12-31'))
    .otherwise(df.prd_end_dt)
)

In [0]:
for col in df.columns:
    print(f'Nulls in {col}: {df.filter(F.col(col).isNull()).count()}')

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name,F.trim(F.col(field.name)))

df.show()

In [0]:
df = df.withColumn('Category_ID', F.regexp_replace(F.substring(df.prd_key, 1, 5), '-', '_'))
df = df.withColumn("prd_key", F.substring(df.prd_key, 7, F.length(df.prd_key)))
df.show()

In [0]:
df.select('prd_line').distinct().show()

In [0]:
df = df.withColumn(
    'prd_line',
    F.when(df.prd_line == 'R', 'Road')
    .when(df.prd_line == 'S', 'Other Sales')
    .when(df.prd_line == 'M', 'Mountain')
    .when(df.prd_line == 'T', 'Touring')
    .otherwise(df.prd_line)
)
df.show()

In [0]:
for old_names, new_names in zip(df.columns[:-1], ['Product_key','Product_name','Product_cost','Product_line','Product_start_date','Product_end_date']):
    df = df.withColumnRenamed(old_names, new_names)
df.show()

In [0]:
df.printSchema()

In [0]:
df.write.mode('overwrite').saveAsTable('bikes_catalog.silver.silver_prd_info')